# Chapter 29: Improve and Writer Workflow


In [ ]:
import sys
from pathlib import Path

ROOT = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "content" / "books" / "retrieval" / "_index.md").is_file():
        ROOT = candidate
        break
if ROOT is None:
    raise FileNotFoundError("repository root not found")

NOTEBOOK_DIR = ROOT / "notebooks" / "books" / "retrieval"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
if str(ROOT / "examples") not in sys.path:
    sys.path.insert(0, str(ROOT / "examples"))

import _support
ROOT = _support.repo_root()
print("repo root:", ROOT)


In [ ]:

# Local mock manuscript-review workflow; no live Writer/Improve integration.
manuscript = [
    {"chapter": 26, "text": "Retrieval cannot be claimed to improve the task from this run."},
    {"chapter": 27, "text": "The contract refuses changed-view references before prompt construction."},
]
sources = [
    {"source_id": "research:consumer", "text": "All 140 task-condition generations failed the frozen answer-and-citation contract."},
    {"source_id": "research:contract", "text": "Hashes detect inconsistency; they are not signatures."},
]
print("manuscript chapters:", [m["chapter"] for m in manuscript])


In [ ]:

# Search -> admission -> proposed change -> review state.
query = "contract"
search_hits = [s for s in sources if query in s["text"].lower() or "contract" in s["text"].lower()]
print("search hits:", [s["source_id"] for s in search_hits])
assert search_hits

admitted = [{"label": "E1", "source_id": search_hits[0]["source_id"], "text": search_hits[0]["text"]}]
print("admitted:", admitted)

proposal = {
    "proposal_id": "P1",
    "target_chapter": 27,
    "status": "proposed",
    "requires_human_approval": True,
    "writes": [],
    "evidence": admitted,
}
print("proposal status:", proposal["status"])
assert proposal["status"] == "proposed"
assert proposal["writes"] == []


In [ ]:

# Deterministic approval gating.
def can_apply(proposal, approved_by_human):
    if not approved_by_human:
        return False, "human_approval_required"
    if proposal["status"] != "proposed":
        return False, "invalid_state"
    return True, "approved_for_application"

ok, reason = can_apply(proposal, False)
print("without approval:", ok, reason)
assert ok is False and reason == "human_approval_required"

ok2, reason2 = can_apply(proposal, True)
print("with approval:", ok2, reason2)
assert ok2 is True
assert proposal["writes"] == []
print("approval gating deterministic; no writes performed")


In [ ]:

# No live Writer/Improve integration or APIs.
application = {
    "Writer": "not integrated",
    "Improve": "not integrated",
    "writes": [],
    "human_approval": "not requested or simulated",
}
print(application)
assert application["Writer"] == "not integrated"
assert application["Improve"] == "not integrated"
assert application["writes"] == []


## Observed versus not established
Observed: a local mock review workflow with deterministic approval gating and no writes. Not established: any Writer/Improve integration, API, database change, or applied patch.